# 정부부처 보도자료 모아보기 (Colab 실행용)

기후에너지환경부 / 산업통상부 보도자료를 가져와 `press_releases.html`을 만듭니다.

**주의**: 무료 Colab은 자동 스케줄 실행을 지원하지 않습니다. 매일 자동으로 돌리려면 GitHub Actions 버전을 사용하세요.
위에서 아래로 셀을 순서대로 실행(▶)하면 됩니다.

In [ ]:
!pip install -q requests beautifulsoup4

## 1. 페이지 디자인 템플릿

In [ ]:
TEMPLATE_HTML = r"""<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>정부부처 보도자료 모아보기</title>
<style>
  :root{
    --ink:#1b2430;
    --paper:#f6f4ee;
    --line:#d9d3c4;
    --navy:#22304a;
    --navy-deep:#141c2c;
    --accent:#b4372b;
    --muted:#6b7280;
    --card:#ffffff;
  }
  *{box-sizing:border-box;}
  body{
    margin:0;
    background:var(--paper);
    color:var(--ink);
    font-family:"Pretendard","Noto Sans KR","Apple SD Gothic Neo",sans-serif;
    line-height:1.5;
  }
  .masthead{
    background:var(--navy-deep);
    color:#f2ede0;
    padding:28px 20px 22px;
    border-bottom:3px solid var(--accent);
  }
  .masthead-inner{
    max-width:860px;
    margin:0 auto;
    display:flex;
    justify-content:space-between;
    align-items:flex-end;
    flex-wrap:wrap;
    gap:10px;
  }
  .masthead h1{
    font-family:"Noto Serif KR","Nanum Myeongjo",serif;
    font-size:26px;
    font-weight:700;
    margin:0 0 4px;
    letter-spacing:-0.01em;
  }
  .masthead .sub{
    font-size:13px;
    color:#b9c2d4;
    letter-spacing:0.02em;
  }
  .masthead .meta{
    font-size:12px;
    color:#8a93a8;
    text-align:right;
    font-variant-numeric:tabular-nums;
  }
  main{
    max-width:860px;
    margin:0 auto;
    padding:26px 20px 60px;
  }
  .section-label{
    display:flex;
    align-items:center;
    gap:10px;
    margin:0 0 14px;
  }
  .section-label .tag{
    background:var(--accent);
    color:#fff;
    font-size:12px;
    font-weight:700;
    padding:3px 9px;
    border-radius:2px;
    letter-spacing:0.03em;
  }
  .section-label h2{
    font-size:16px;
    margin:0;
    font-weight:700;
    color:var(--navy-deep);
  }
  .section-label .count{
    font-size:12px;
    color:var(--muted);
    font-variant-numeric:tabular-nums;
  }
  ul.list{
    list-style:none;
    margin:0 0 30px;
    padding:0;
    border-top:1px solid var(--line);
  }
  ul.list li{
    border-bottom:1px solid var(--line);
  }
  ul.list a{
    display:flex;
    gap:14px;
    align-items:baseline;
    padding:13px 4px;
    text-decoration:none;
    color:var(--ink);
  }
  ul.list a:hover .title{
    color:var(--accent);
    text-decoration:underline;
  }
  .src{
    flex:0 0 auto;
    font-size:11px;
    font-weight:700;
    padding:3px 7px;
    border-radius:2px;
    color:#fff;
    white-space:nowrap;
  }
  .src.mcee{background:#2f6b4f;}
  .src.motir{background:#2a4d8f;}
  .title{
    flex:1 1 auto;
    font-size:14.5px;
  }
  .date{
    flex:0 0 auto;
    font-size:12px;
    color:var(--muted);
    font-variant-numeric:tabular-nums;
  }
  .empty{
    padding:26px 4px;
    color:var(--muted);
    font-size:13.5px;
    border-top:1px solid var(--line);
    border-bottom:1px solid var(--line);
    margin-bottom:30px;
  }
  .more-toggle{
    display:block;
    width:100%;
    background:var(--navy);
    color:#f2ede0;
    border:none;
    border-radius:3px;
    padding:12px;
    font-size:13.5px;
    font-weight:600;
    cursor:pointer;
    letter-spacing:0.01em;
  }
  .more-toggle:hover{background:var(--navy-deep);}
  #recent-wrap{display:none; margin-top:18px;}
  #recent-wrap.open{display:block;}
  footer{
    max-width:860px;
    margin:0 auto;
    padding:0 20px 40px;
    font-size:11.5px;
    color:var(--muted);
  }
  @media (max-width:520px){
    .masthead h1{font-size:21px;}
    .title{font-size:13.5px;}
  }
</style>
</head>
<body>

<div class="masthead">
  <div class="masthead-inner">
    <div>
      <h1>정부부처 보도자료 모아보기</h1>
      <div class="sub">기후에너지환경부 · 산업통상부</div>
    </div>
    <div class="meta" id="meta-info"></div>
  </div>
</div>

<main>
  <div class="section-label">
    <span class="tag">오늘</span>
    <h2 id="today-label">오늘의 보도자료</h2>
    <span class="count" id="today-count"></span>
  </div>
  <div id="today-wrap"></div>

  <button class="more-toggle" id="more-btn">최근 보도자료 10건 더보기</button>

  <div id="recent-wrap">
    <div class="section-label" style="margin-top:26px;">
      <span class="tag" style="background:var(--navy);">최근</span>
      <h2>최근 게시물</h2>
      <span class="count" id="recent-count"></span>
    </div>
    <ul class="list" id="recent-list"></ul>
  </div>
</main>

<footer>
  본 페이지는 각 부처 홈페이지의 보도자료 게시판을 자동 수집해 제목과 원문 링크만 정리한 것입니다.
  기사 원문 및 첨부파일은 링크를 눌러 각 부처 공식 홈페이지에서 확인해 주세요.
</footer>

<script id="press-data" type="application/json">__DATA__</script>
<script>
  const data = JSON.parse(document.getElementById('press-data').textContent);

  document.getElementById('meta-info').textContent =
    `${data.today} 기준 · 생성 ${data.generated_at}`;

  function srcClass(name){
    if(name.includes('기후')) return 'mcee';
    if(name.includes('산업')) return 'motir';
    return '';
  }

  function renderList(items){
    if(!items.length) return null;
    const ul = document.createElement('ul');
    ul.className = 'list';
    items.forEach(it => {
      const li = document.createElement('li');
      const a = document.createElement('a');
      a.href = it.url;
      a.target = '_blank';
      a.rel = 'noopener noreferrer';
      a.innerHTML =
        `<span class="src ${srcClass(it.source)}">${it.source}</span>` +
        `<span class="title">${it.title}</span>` +
        `<span class="date">${it.date || ''}</span>`;
      li.appendChild(a);
      ul.appendChild(li);
    });
    return ul;
  }

  // 오늘의 보도자료
  const todayWrap = document.getElementById('today-wrap');
  document.getElementById('today-count').textContent = `${data.today_items.length}건`;
  const todayList = renderList(data.today_items);
  if(todayList){
    todayWrap.appendChild(todayList);
  } else {
    const empty = document.createElement('div');
    empty.className = 'empty';
    empty.textContent = '오늘 등록된 보도자료가 아직 없습니다. 아래 "더보기"에서 최근 게시물을 확인하세요.';
    todayWrap.appendChild(empty);
  }

  // 최근 게시물
  document.getElementById('recent-count').textContent = `${data.recent_items.length}건`;
  const recentList = renderList(data.recent_items);
  const recentListEl = document.getElementById('recent-list');
  if(recentList){
    recentListEl.replaceWith(recentList);
    recentList.id = 'recent-list';
  }

  document.getElementById('more-btn').addEventListener('click', function(){
    const wrap = document.getElementById('recent-wrap');
    const open = wrap.classList.toggle('open');
    this.textContent = open ? '최근 게시물 접기' : '최근 보도자료 10건 더보기';
  });
</script>

</body>
</html>
"""

## 2. 수집 + 생성

In [ ]:
import json, re, sys
from datetime import datetime
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"}
DATE_RE = re.compile(r"\d{4}-\d{2}-\d{2}")

SITES = [
    {"name": "기후에너지환경부", "list_url": "https://mcee.go.kr/home/web/index.do?menuId=10598", "base": "https://mcee.go.kr", "parser": "mcee"},
    {"name": "산업통상부", "list_url": "https://www.motir.go.kr/kor/article/ATCL3f49a5a8c", "base": "https://www.motir.go.kr", "parser": "motir"},
]

def fetch(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding or resp.encoding
    return resp.text

def find_main_table(soup):
    for table in soup.find_all("table"):
        if (table.find("a", href=True) or table.find("a", onclick=True)) and len(table.find_all("tr")) >= 3:
            return table
    return soup.find("table")

def extract_date(row):
    for cell in row.find_all(["td", "th"]):
        text = cell.get_text(strip=True)
        if DATE_RE.match(text):
            return DATE_RE.match(text).group(0)
    return ""

def parse_mcee(html, base):
    soup = BeautifulSoup(html, "html.parser")
    table = find_main_table(soup)
    items = []
    if not table:
        return items
    for row in table.find_all("tr"):
        a = row.find("a", href=True)
        if not a or "read.do" not in a["href"]:
            continue
        title = a.get_text(strip=True)
        if not title:
            continue
        href = a["href"]
        if href.startswith("/"):
            href = base + href
        items.append({"title": title, "url": href, "date": extract_date(row)})
    return items

def parse_motir(html, base):
    soup = BeautifulSoup(html, "html.parser")
    table = find_main_table(soup)
    items = []
    if not table:
        return items
    list_url_match = re.search(r"/kor/article/(ATCL[0-9a-fA-F]+)", html)
    for row in table.find_all("tr"):
        a = row.find("a", href=True) or row.find("a")
        if not a:
            continue
        href = a.get("href", "") or a.get("onclick", "")
        m = re.search(r"article\.view\(\'(\d+)\'\)", href)
        if not m:
            continue
        article_id = m.group(1)
        title = a.get_text(strip=True)
        if not title:
            continue
        board_code = list_url_match.group(1) if list_url_match else "ATCL3f49a5a8c"
        url = f"{base}/kor/article/{board_code}/{article_id}/view"
        items.append({"title": title, "url": url, "date": extract_date(row)})
    return items

PARSERS = {"mcee": parse_mcee, "motir": parse_motir}

today = datetime.now().strftime("%Y-%m-%d")
all_items = []
for site in SITES:
    try:
        html = fetch(site["list_url"])
    except Exception as exc:
        print("[경고]", site["name"], "접속 실패:", exc)
        continue
    parsed = PARSERS[site["parser"]](html, site["base"])
    for item in parsed:
        item["source"] = site["name"]
    all_items.extend(parsed)
    print(site["name"] + ":", len(parsed), "건 수집")

all_items.sort(key=lambda x: x["date"] or "0000-00-00", reverse=True)
today_items = [i for i in all_items if i["date"] == today]
recent_items = all_items[:10]

data = {
    "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "today": today,
    "today_items": today_items,
    "recent_items": recent_items,
}

html_out = TEMPLATE_HTML.replace("__DATA__", json.dumps(data, ensure_ascii=False))
with open("press_releases.html", "w", encoding="utf-8") as f:
    f.write(html_out)

print(f"완료: press_releases.html 생성 (오늘자 {len(today_items)}건 / 최근 {len(recent_items)}건)")


## 3. 결과 파일 다운로드

In [ ]:
from google.colab import files
files.download('press_releases.html')